In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import string

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model hyperparameters - make sure these match the training configuration
n_embd = 256
block_size = 256
n_head = 8
n_layer = 6
dropout = 0.2

# Transformer components
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ Linear layer followed by non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual connections around each component
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate text with optional temperature sampling"""
        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -block_size:] if idx.size(1) > block_size else idx
            logits, _ = self(idx_cond, None)
            # Focus only on the last step's predictions
            logits = logits[:, -1, :] / temperature  # Apply temperature
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

def load_model(model_path):
    """Load the saved model and vocabulary"""
    print(f"Loading model from {model_path}...")
    
    checkpoint = torch.load(model_path, map_location=device)
    char_to_idx = checkpoint['char_to_idx']
    idx_to_char = checkpoint['idx_to_char']
    vocab_size = len(char_to_idx)
    
    model = GPTLanguageModel(vocab_size).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to evaluation mode
    
    print(f"Model loaded successfully with vocabulary size: {vocab_size}")
    
    return model, char_to_idx, idx_to_char

def generate_poem(model, char_to_idx, idx_to_char, prompt, max_tokens=400, temperature=0.8):
    """Generate a poem starting with the given prompt"""
    # Convert prompt to indices
    initial_indices = [char_to_idx.get(char, 0) for char in prompt]
    context = torch.tensor(initial_indices, dtype=torch.long, device=device).unsqueeze(0)
    
    # Generate text
    with torch.no_grad():  # No need to track gradients during generation
        generated = model.generate(context, max_new_tokens=max_tokens, temperature=temperature)
    
    # Convert indices back to characters
    generated_text = ''.join([idx_to_char[idx.item()] for idx in generated[0]])
    
    return generated_text

def main():
    # Path to the saved model
    model_path = 'poem_gen_model.pth'
    
    # Load the model
    model, char_to_idx, idx_to_char = load_model(model_path)
    
    # Define prompts or get from user
    while True:
        print("\nPoem Generator Options:")
        print("1. Use a pre-defined prompt")
        print("2. Enter your own prompt")
        print("3. Quit")
        
        choice = input("Select an option (1-3): ")
        
        if choice == '1':
            prompts = [
                "Love", 
                "The moonlight", 
                "In the garden", 
                "Dreams of", 
                "Once upon a time"
            ]
            
            print("\nAvailable prompts:")
            for i, prompt in enumerate(prompts, 1):
                print(f"{i}. {prompt}")
            
            prompt_choice = input(f"Select a prompt (1-{len(prompts)}): ")
            try:
                prompt = prompts[int(prompt_choice) - 1]
            except (ValueError, IndexError):
                print("Invalid choice. Using 'The moonlight' as default.")
                prompt = "The moonlight"
                
        elif choice == '2':
            prompt = input("Enter your prompt: ")
            if not prompt:
                prompt = "Poetry"
                print(f"Using default prompt: '{prompt}'")
                
        elif choice == '3':
            print("Goodbye!")
            break
            
        else:
            print("Invalid choice. Please try again.")
            continue
        
        # Let user customize generation parameters
        try:
            max_length = int(input("Enter maximum length (default: 400): ") or "400")
        except ValueError:
            max_length = 400
            print(f"Using default length: {max_length}")
            
        try:
            temp = float(input("Enter temperature (0.1-2.0, default: 0.8): ") or "0.8")
            # Constrain temperature to reasonable values
            temp = max(0.1, min(2.0, temp))
        except ValueError:
            temp = 0.8
            print(f"Using default temperature: {temp}")
        
        # Generate the poem
        print(f"\n--- Generating poem with prompt '{prompt}', length {max_length}, temperature {temp} ---")
        poem = generate_poem(model, char_to_idx, idx_to_char, prompt, max_length, temp)
        
        print("\n" + poem + "\n")
        print("-" * 50)

if __name__ == "__main__":
    main()

Using device: cuda
Loading model from D:\story_poem\finetuned_poem_model_final.pth...


C:\Users\Manan\AppData\Local\Temp\ipykernel_59124\4011823512.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded successfully with vocabulary size: 38

Poem Generator Options:
1. Use a pre-defined prompt
2. Enter your own prompt
3. Quit

--- Generating poem with prompt 'Love is life', length 100, temperature 1.0 ---


ove is life and slap my hands
in me, and ties of all that invite,
and sees the hand of his course.
for there's 

--------------------------------------------------

Poem Generator Options:
1. Use a pre-defined prompt
2. Enter your own prompt
3. Quit

--- Generating poem with prompt 'i went to nature ', length 400, temperature 1.0 ---

i went to nature forward,
wherever i did not make all you said.
and they must not say
that one that i reply
that you were also for such vaunt recolt,
why, bestowed iup and declared;
and set in a grecian common court,
and though long song in air,
and it was
the depth, the monster's pipe,
a three times rang
not to pity afar,
of all this proofs destiny.
he thought if my miracle made,
though heavy past,
where i took h

------------------------